In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
path = sorted(Path('../data/snapshots').glob('shortages_*.csv'))[-1]
df = pd.read_csv(path)
print(f'Size of df: {len(df)}')

df = df.rename(columns={
    'Bearbeitungsnummer' : 'report_id',
    'PZN': 'pzn',
    'ENR': 'enr',
    'Anm. zum Grund': 'reason_note',
    'Info an Fachkreise': 'professionals_informed',
    'Beginn' : 'start_date',
    'Ende' : 'end_date',
    'Grund' : 'reason',
    'Krankenhausrelevant': 'hospital_relevant',
    'Atc Code': 'atc_code',
    'Meldungsart': 'report_type',
    'Datum der letzten Meldung': 'last_report_date',
    'Art des Grundes': 'reason_type',
    'Zulassungsinhaber': 'licensee',
    'klassifikation': 'classification',
    'Alternativpräparat': 'alternative_product',
    'Referenzierte Erstmeldung': 'initial_report_id',
    'Arzneimittlbezeichnung': 'drug_name',
    'Wirkstoffe': 'active_ingredients',
    'Darreichungsform': 'dosage_form',
    'Datum der Erstmeldung': 'initial_report_date'
})

date_cols = [x for x in df.columns if 'date' in x]

assert len(date_cols) == 4

for col in date_cols:
    df[col] = pd.to_datetime(df[col], format='%d.%m.%Y')

df.loc[
    df['licensee'].str.contains(r'^h e x a l.*$', case=False, na=False),
    'licensee'
] = 'Hexal AG'

assert not df['licensee'].str.contains(r'^h e x a l.*$', case=False, na=False).any()

print(df.dtypes)
display(df.head())
print(df['report_id'].nunique())

Size of df: 936
pzn                                int64
enr                                  str
report_id                            str
initial_report_id                    str
report_type                          str
start_date                datetime64[us]
end_date                  datetime64[us]
last_report_date          datetime64[us]
reason_type                          str
drug_name                            str
atc_code                             str
active_ingredients                   str
hospital_relevant                    str
licensee                             str
reason                               str
reason_note                          str
alternative_product                  str
initial_report_date       datetime64[us]
professionals_informed               str
dosage_form                          str
classification                       str
dtype: object


,pzn,enr,report_id,initial_report_id,report_type,start_date,end_date,last_report_date,reason_type,drug_name,atc_code,active_ingredients,hospital_relevant,licensee,reason,reason_note,alternative_product,initial_report_date,professionals_informed,dosage_form,classification
0,11213101,2113068,LE2026004167,LE2026003864,Änderungsmeldung,2026-10-14,2026-12-27,2026-11-13,Sonstige,Volon A 40,H02AB08,Triamcinolonacetonid,ja,Dermapharm Aktiengesellschaft,Erhöhte Nachfrage,NaN,NaN,2026-10-20,Nein,Kristallsuspension,versrel
1,16804143,2163020,LE2026004166,LE2025001314,Änderungsmeldung,2025-04-14,2026-12-01,2026-11-12,Sonstige,Sertralin TAD 50 mg Filmtabletten,N06AB06,Sertralinhydrochlorid,ja,TAD Pharma GmbH,Sonstige,NaN,NaN,2025-04-14,Nein,Filmtablette,weder versrel noch verskri
2,16804172,2163020,LE2026004166,LE2025001314,Änderungsmeldung,2025-04-14,2026-12-01,2026-11-12,Sonstige,Sertralin TAD 50 mg Filmtabletten,N06AB06,Sertralinhydrochlorid,ja,TAD Pharma GmbH,Sonstige,NaN,NaN,2025-04-14,Nein,Filmtablette,weder versrel noch verskri
3,16804189,2163020,LE2026004166,LE2025001314,Änderungsmeldung,2025-04-14,2026-12-01,2026-11-12,Sonstige,Sertralin TAD 50 mg Filmtabletten,N06AB06,Sertralinhydrochlorid,ja,TAD Pharma GmbH,Sonstige,NaN,NaN,2025-04-14,Nein,Filmtablette,weder versrel noch verskri
4,19147214,7011798,LE2026004163,NaN,Erstmeldung,2026-10-08,2026-12-15,2026-11-12,Sonstige,EXBLIFEP,J01DE51,Cefepimdihydrochlorid-Monohydrat; Enmetazobactam,ja,Advanz Pharma Limited,Sonstige,NaN,NaN,2026-11-12,Vorgesehen,Pulver für ein Konzentrat zur Herstellung eine...,verskri (auch versrel)


532


In [3]:
hospital_map = {'ja': True, 'nein': False}

report_map = {'Erstmeldung' : 'initial', 'Änderungsmeldung': 'update'}

reason_map = {
    'Erhöhte Nachfrage': 'Increased demand',
    'Sonstige': 'Other',
    'Probleme in der Herstellung': 'Manufacturing problems',
    'Probleme bei der Endfreigabe': 'Final batch release issues',
    'Herstellerwechsel': 'Change of manufacturer',
    'GMP Mängel': 'GMP deficiencies',
    'Probleme bei der Chargenprüfung': 'Batch testing issues',
    'Probleme beim Wirkstoffhersteller': 'Active ingredient manufacturer issues',
    'Probleme beim Sonstigen Hersteller': 'Issues at another manufacturer',
    'Unzureichende Produktionskapazitäten': 'Insufficient production capacity',
    'Änderung des Herstellverfahrens': 'Change in manufacturing process'
}

professionals_map = {
    'Ja' : 'yes',
    'Nein' : 'no',
    'Vorgesehen' : 'planned'
}

reason_type_map = {
    'Sonstige' : 'other',
    'Produktionsproblem' : 'production_problem'
}

classification_map = {
    'versrel' : 'supply_relevant',
    'verskri (auch versrel)' : 'supply_critical',
    'weder versrel noch verskri': 'neither'
}

d = {
    'hospital_relevant': hospital_map,
    'report_type': report_map,
    'reason' : reason_map,
    'classification' : classification_map,
    'reason_type' : reason_type_map,
    'professionals_informed': professionals_map
}

for col, dic in d.items():
    assert set(df[col].dropna()) <= set(dic)
    df[col] = df[col].replace(dic)

In [4]:
pzn_missing = df['pzn'] == 0
print(pzn_missing.sum())

df.loc[pzn_missing, 'pzn'] = np.nan

df['pzn'] = df['pzn'].astype('Int64')

atc_missing = df['atc_code'].str.startswith('TERM_ID_NA') | (df['atc_code'] == 'XXXXXXX')
print(atc_missing.sum())

df.loc[atc_missing, 'atc_code'] = np.nan

code = df['atc_code'].str.split(',').str[0].str.strip().str[0]

atc_group_map = {
    'A' : 'Alimentary tract and metabolism',
    'B' : 'Blood',
    'C' : 'Cardiovascular system',
    'D' : 'Dermatologicals',
    'G' : 'Genito-urinary system and sex hormones',
    'H' : 'Systemic hormones',
    'J' : 'Anti-infectives',
    'L' : 'Antineoplastic and immunomodulating agents',
    'M' : 'Musculo-skeletal system',
    'N' : 'Nervous system',
    'P' : 'Antiparasitic products',
    'R' : 'Respiratory system',
    'S' : 'Sensory organs',
    'V' : 'Various'
}

assert set(code.dropna()) <= set(atc_group_map)
df['atc_group'] = code.map(atc_group_map)

assert df['atc_group'].notna().sum() == df['atc_code'].notna().sum()
reports = df.drop_duplicates(subset='report_id')
assert len(reports) == df['report_id'].nunique()

n_packages = df.groupby('report_id').size().rename('n_packages')
assert n_packages.sum() == len(df)

reports = reports.merge(
    n_packages,
    how='left',
    on='report_id',
    validate='one_to_one'
)

reports = reports.drop(columns=['pzn'])

8
22


In [5]:
reports['atc_group'].value_counts(dropna=False)

atc_group
Nervous system                                159
Cardiovascular system                          74
Antineoplastic and immunomodulating agents     64
Anti-infectives                                50
Alimentary tract and metabolism                35
Blood                                          26
Systemic hormones                              24
Sensory organs                                 21
Genito-urinary system and sex hormones         18
Various                                        14
Dermatologicals                                14
NaN                                            11
Musculo-skeletal system                        11
Respiratory system                             10
Antiparasitic products                          1
Name: count, dtype: int64

In [6]:
df.to_parquet('../data/processed/packages.parquet', index=False)
reports.to_parquet('../data/processed/reports.parquet', index=False)